# EXP-01 — Baseline ResNet50 224px (CBIS-DDSM)
## Execução em Kaggle com tracking no MLflow

Notebook equivalente ao comando local/Docker (`train.py --config config_exp01.yaml`
+ `predict.py`), com rastreamento de experimentos no MLflow seguindo o
**Padrão de tracking — MLflow** (tags, parâmetros, métricas e artefatos
obrigatórios).

**Dependências:** S3-01, S3-02 e configuração mínima do MLflow (servidor já
provisionado — este notebook só se conecta a ele via `MLFLOW_TRACKING_URI`,
não configura o servidor).

**Fluxo deste notebook:**
1. Checklist de entrada (dataset/manifest, split, seed, config, ambiente)
2. Conexão ao MLflow (credenciais via Kaggle Secrets, nunca hardcoded)
3. **Smoke test** — obrigatório rodar e passar antes do run completo
4. Run completo — treino + predição no teste congelado + métricas com IC 95%

**Segurança de dados (crítico):** nenhum arquivo DICOM, imagem clínica,
caminho pessoal de máquina ou credencial é publicado neste notebook ou no
seu output. Checkpoints e CSVs de predição vão *apenas* para o MLflow
(artifact store próprio, não o output público do Kaggle) e são apagados da
pasta de trabalho imediatamente após o upload.

## Checklist de entrada (leitura manual antes de rodar)

Confirme cada item **antes** de rodar a célula do smoke test. A célula de
código logo abaixo também valida boa parte disso automaticamente, mas a
leitura manual continua sendo a checagem principal.

| Item | Confirmação |
|---|---|
| **Dataset / manifest** | O dataset CBIS-DDSM está anexado ao notebook (`/kaggle/input/...`) e os 4 CSVs oficiais (mass/calc × train/test) existem no caminho configurado |
| **Split** | `train_csvs` e `test_csvs` apontam para arquivos diferentes — o teste é o split oficial do CBIS-DDSM, nunca re-sorteado |
| **Seed** | `CONFIG["seed"]` está definida e é a mesma usada em runs anteriores comparáveis |
| **Configuração** | `CONFIG` revisado: arquitetura, resolução, batch, LR, épocas correspondem ao experimento pretendido (ex: EXP-01 na matriz de experimentos) |
| **Ambiente** | Rodando em Kaggle (ou nuvem aprovada) com GPU do tipo esperado — confira em Settings > Accelerator |
| **Segredos MLflow** | `MLFLOW_TRACKING_URI` (e token, se aplicável) cadastrados em Add-ons > Secrets — nunca colados no código |
| **Orçamento de GPU** | Custo estimado deste run já checado contra o limite semanal (ver planilha de plano de execução) antes de rodar o run completo |


In [1]:
import os

base = "/kaggle/input/datasets"
for root, dirs, files in os.walk(base):
    depth = root.replace(base, '').count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print(root)
    for f in files[:5]:
        print("   ", f)

/kaggle/input/datasets
/kaggle/input/datasets/orvile
/kaggle/input/datasets/orvile/cbis-ddsm-dataset
    LICENSE.md
    README.md
    meta.json
/kaggle/input/datasets/orvile/cbis-ddsm-dataset/test
/kaggle/input/datasets/orvile/cbis-ddsm-dataset/train


In [ ]:
import os, sys, subprocess, json, hashlib, time
from datetime import datetime

# ============================================================
# CONFIG — edite apenas estes valores antes de rodar.
# NUNCA coloque caminho pessoal de máquina local aqui (ex: C:\Users\...).
# Neste notebook, os dados vêm sempre de /kaggle/input/... (dataset Kaggle).
# ============================================================
CONFIG = {
    "experiment_id": "EXP-01",
    "owner": "Guilherme BettegaS",          # <-- troque pelo seu usuário/identificador de papel
    "role": "02",                     # <-- número do papel responsável (ver padrão de tags)
    "ablation_axis": "resolution",    # eixo variado nesta rodada (baseline = ponto de partida do eixo "resolution")
    "seed": 42,
    "architecture": "resnet50",
    "pretrained_weights": "imagenet",
    "image_size": 224,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "epochs": 20,
    "dataset_root": "/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset",
    "train_csvs": [
        "csv/mass_case_description_train_set.csv",
        "csv/calc_case_description_train_set.csv",
    ],
    "test_csvs": [
        "csv/mass_case_description_test_set.csv",
        "csv/calc_case_description_test_set.csv",
    ],
    "image_path_column": "cropped image file path",
    "patient_id_column": "patient_id",
    "pathology_column": "pathology",
    "malignant_labels": ["MALIGNANT"],
    "non_malignant_labels": ["BENIGN", "BENIGN_WITHOUT_CALLBACK"],
    "val_fraction": 0.15,
    "preprocessing_version": "preprocessing@REPLACE_WITH_COMMIT_OR_TAG",
}

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)


def get_code_version():
    """Hash do commit Git ativo. Em notebooks Kaggle não há .git local por
    padrão — se este notebook foi gerado a partir de um script versionado,
    defina a variável de ambiente CODE_VERSION antes de rodar (ex: via
    Kaggle Secrets ou como parâmetro do kernel). Caso contrário, cai para
    um placeholder explícito (NUNCA deixe isso como 'unknown' num run real)."""
    env_version = os.environ.get("CODE_VERSION")
    if env_version:
        return env_version
    try:
        out = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL)
        return out.decode().strip()
    except Exception:
        return "UNKNOWN_SET_CODE_VERSION_ENV_VAR"


def detect_environment():
    """Detecta o ambiente de execução para a tag `environment`."""
    if os.path.exists("/kaggle"):
        try:
            gpu_info = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                stderr=subprocess.DEVNULL,
            ).decode().strip()
            return "kaggle-gpu" if gpu_info else "kaggle-cpu"
        except Exception:
            return "kaggle-cpu"
    return "local-cpu"


CODE_VERSION = get_code_version()
ENVIRONMENT = detect_environment()

print(f"CODE_VERSION: {CODE_VERSION}")
print(f"ENVIRONMENT detectado: {ENVIRONMENT}")


### Checklist de entrada — validação automática

In [ ]:
# ============================================================
# CHECKLIST DE ENTRADA — roda automaticamente, mas NÃO substitui a leitura
# manual da checklist.md antes de aprovar a execução completa.
# Falha (assert) se algo obrigatório estiver faltando, em vez de deixar o
# treino começar com uma configuração incompleta.
# ============================================================

checklist_status = {}

# 1. Dataset / manifest presente
dataset_ok = os.path.isdir(CONFIG["dataset_root"])
checklist_status["dataset_root_existe"] = dataset_ok

manifest_files_ok = []
for csv_list_name in ["train_csvs", "test_csvs"]:
    for rel_path in CONFIG[csv_list_name]:
        full_path = os.path.join(CONFIG["dataset_root"], rel_path)
        manifest_files_ok.append(os.path.exists(full_path))
checklist_status["todos_os_csvs_encontrados"] = all(manifest_files_ok)

# 2. Split — confirma que vamos usar o split OFICIAL (arquivos de teste
# separados dos de treino, nunca a mesma lista)
checklist_status["split_train_test_distintos"] = (
    set(CONFIG["train_csvs"]).isdisjoint(set(CONFIG["test_csvs"]))
)

# 3. Seed definida
checklist_status["seed_definida"] = isinstance(CONFIG["seed"], int)

# 4. Configuração sem caminho pessoal de máquina local
config_str = json.dumps(CONFIG)
personal_path_patterns = ["C:\\Users", "/home/", "/Users/"]
checklist_status["sem_caminho_pessoal_no_config"] = not any(
    p in config_str for p in personal_path_patterns
)

# 5. Ambiente detectado é um ambiente aprovado (Kaggle ou nuvem aprovada)
checklist_status["ambiente_aprovado"] = ENVIRONMENT.startswith("kaggle") or ENVIRONMENT == "docker"

# 6. code_version não é o placeholder de erro
checklist_status["code_version_definida"] = CODE_VERSION != "UNKNOWN_SET_CODE_VERSION_ENV_VAR"

print("=== CHECKLIST DE ENTRADA ===")
for item, ok in checklist_status.items():
    print(f"  [{'OK' if ok else 'FALTA'}] {item}")

obrigatorios = ["dataset_root_existe", "todos_os_csvs_encontrados",
                 "split_train_test_distintos", "seed_definida",
                 "sem_caminho_pessoal_no_config", "ambiente_aprovado"]
faltando = [k for k in obrigatorios if not checklist_status[k]]

if faltando:
    raise RuntimeError(
        f"Checklist de entrada incompleto, itens faltando: {faltando}. "
        "Corrija antes de rodar o smoke test ou o treino completo."
    )

if not checklist_status["code_version_definida"]:
    print("\n[AVISO] CODE_VERSION não definida via variável de ambiente. "
          "Defina antes do run completo (não bloqueante para o smoke test).")

print("\nChecklist obrigatório: OK. Pode prosseguir para o smoke test.")


### Dataset — carregamento dos CSVs oficiais e split por paciente

In [ ]:
# dataset.py
"""Dataset do CBIS-DDSM para classificação binária maligno x não maligno em
patches (imagens já recortadas na ROI da lesão).

CORREÇÃO IMPORTANTE (descoberta ao rodar no Kaggle): a coluna
'cropped image file path' dos CSVs mass/calc_case_description_*.csv aponta
para o caminho do arquivo DICOM ORIGINAL (.dcm), que não existe mais nesse
download — só as versões .jpg convertidas existem, dentro de jpeg/<SeriesInstanceUID>/.
Para achar o .jpg certo, usamos o csv/dicom_info.csv como tabela de tradução:
ele tem uma linha por imagem, com 'SeriesInstanceUID' e 'image_path' (o
caminho relativo real do .jpg, tipo 'CBIS-DDSM/jpeg/<UID>/1-172.jpg').

A tradução funciona assim:
    1. Extrai o SeriesInstanceUID do caminho bruto (.dcm) do CSV de casos
       — é o penúltimo segmento do caminho, logo antes do nome do arquivo.
    2. Procura esse SeriesInstanceUID no dicom_info.csv.
    3. Usa o 'image_path' correspondente (já aponta pro .jpg real).
"""

import os
import re
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


# Um UID de DICOM é uma sequência de números separados por ponto (ex:
# "1.3.6.1.4.1.9590.100.1.2.128573...").
_UID_PATTERN = re.compile(r"^\d+(\.\d+)+$")


def load_and_merge_csvs(dataset_root: str, csv_paths: List[str]) -> pd.DataFrame:
    dfs = []
    for rel_path in csv_paths:
        full_path = os.path.join(dataset_root, rel_path)
        if not os.path.exists(full_path):
            raise FileNotFoundError(
                f"CSV não encontrado: {full_path}\n"
                "Confira 'dataset_root' e 'train_csvs'/'test_csvs' no config."
            )
        df = pd.read_csv(full_path)
        df.columns = [c.strip() for c in df.columns]
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def label_from_pathology(pathology: str, malignant_labels, non_malignant_labels) -> Optional[int]:
    pathology = str(pathology).strip().upper()
    if pathology in [p.upper() for p in malignant_labels]:
        return 1
    if pathology in [p.upper() for p in non_malignant_labels]:
        return 0
    return None


def extract_series_uid(raw_path: str) -> Optional[str]:
    """Extrai o SeriesInstanceUID de um caminho bruto do CSV de casos.

    Exemplo de entrada:
      '.../Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1...774/1.3.6.1...846/000000.dcm'
    O SeriesInstanceUID é o segmento de UID imediatamente ANTES do nome do
    arquivo (o mais profundo dos dois UIDs no caminho).
    """
    raw_path = str(raw_path).replace("\\", "/").strip()
    parts = [p for p in raw_path.split("/") if p]
    if len(parts) < 2:
        return None
    # O penúltimo segmento (logo antes do nome do arquivo .dcm) é o SeriesInstanceUID
    candidate = parts[-2]
    if _UID_PATTERN.match(candidate):
        return candidate
    # fallback: procura o último segmento que pareça um UID, de trás pra frente
    for p in reversed(parts[:-1]):
        if _UID_PATTERN.match(p):
            return p
    return None


def build_dicom_info_lookup(dataset_root: str, dicom_info_rel_path: str = "csv/dicom_info.csv") -> dict:
    """Lê o dicom_info.csv e monta um dicionário SeriesInstanceUID -> caminho
    relativo real do .jpg (relativo a dataset_root)."""
    full_path = os.path.join(dataset_root, dicom_info_rel_path)
    if not os.path.exists(full_path):
        raise FileNotFoundError(
            f"dicom_info.csv não encontrado em {full_path} — necessário para "
            "traduzir os caminhos .dcm dos CSVs de casos para os .jpg reais."
        )
    df = pd.read_csv(full_path)
    df.columns = [c.strip() for c in df.columns]

    if "SeriesInstanceUID" not in df.columns or "image_path" not in df.columns:
        raise KeyError(
            f"dicom_info.csv não tem as colunas esperadas ('SeriesInstanceUID', "
            f"'image_path'). Colunas encontradas: {list(df.columns)}"
        )

    lookup = {}
    for _, row in df.iterrows():
        uid = str(row["SeriesInstanceUID"]).strip()
        # image_path vem como 'CBIS-DDSM/jpeg/<uid>/1-172.jpg' — removemos o
        # prefixo 'CBIS-DDSM/' pois nosso dataset_root já aponta pra dentro dela.
        img_path = str(row["image_path"]).strip().replace("\\", "/")
        if img_path.startswith("CBIS-DDSM/"):
            img_path = img_path[len("CBIS-DDSM/"):]
        lookup[uid] = img_path

    return lookup


def resolve_image_path(dataset_root: str, raw_path: str, dicom_lookup: dict) -> str:
    """Traduz o caminho bruto do CSV de casos (.dcm) para o .jpg real,
    usando o dicionário construído a partir do dicom_info.csv."""
    series_uid = extract_series_uid(raw_path)

    if series_uid and series_uid in dicom_lookup:
        return os.path.join(dataset_root, dicom_lookup[series_uid])

    # Fallbacks, caso o caminho já venha correto ou em outro formato:
    raw_path_clean = str(raw_path).replace("\\", "/").strip()
    if os.path.isabs(raw_path_clean) and os.path.exists(raw_path_clean):
        return raw_path_clean

    marker = "jpeg/"
    idx = raw_path_clean.find(marker)
    if idx != -1:
        candidate = os.path.join(dataset_root, raw_path_clean[idx:])
        if os.path.exists(candidate):
            return candidate

    # Última tentativa: junta direto (provavelmente vai falhar e apontar o erro certo)
    return os.path.join(dataset_root, raw_path_clean.lstrip("/"))


class CBISDDSMPatchDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, dataset_root: str, image_path_column: str,
                 resolution: int, normalize_mean, normalize_std, augment: bool = False,
                 dicom_lookup: Optional[dict] = None):
        self.df = dataframe.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.image_path_column = image_path_column
        self.dicom_lookup = dicom_lookup or {}

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((resolution, resolution)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.2),
                transforms.RandomRotation(degrees=10),
                transforms.ToTensor(),
                transforms.Normalize(mean=normalize_mean, std=normalize_std),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((resolution, resolution)),
                transforms.ToTensor(),
                transforms.Normalize(mean=normalize_mean, std=normalize_std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_image_path(self.dataset_root, row[self.image_path_column], self.dicom_lookup)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(
                f"Falha ao abrir imagem '{img_path}' (linha {idx} do dataframe, "
                f"paciente {row.get('patient_id', '?')}). Caminho original no CSV: "
                f"'{row[self.image_path_column]}'. Erro original: {e}"
            )

        img_t = self.transform(img)
        label = torch.tensor(float(row["label"]), dtype=torch.float32)

        return img_t, label, str(row.get("patient_id", "")), img_path


def build_labeled_dataframe(dataset_root: str, csv_paths: List[str], malignant_labels,
                             non_malignant_labels, patient_id_column: str,
                             pathology_column: str) -> pd.DataFrame:
    df = load_and_merge_csvs(dataset_root, csv_paths)

    if patient_id_column not in df.columns:
        raise KeyError(
            f"Coluna '{patient_id_column}' não encontrada no CSV. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    if pathology_column not in df.columns:
        raise KeyError(
            f"Coluna '{pathology_column}' não encontrada no CSV. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    df["label"] = df[pathology_column].apply(
        lambda p: label_from_pathology(p, malignant_labels, non_malignant_labels)
    )
    n_before = len(df)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    n_after = len(df)
    if n_after < n_before:
        print(f"[dataset] Descartadas {n_before - n_after} linhas com rótulo de patologia "
              f"desconhecido (fora de {malignant_labels + non_malignant_labels}).")

    df["patient_id"] = df[patient_id_column]
    return df


def filter_resolvable_rows(df: pd.DataFrame, dataset_root: str, image_path_column: str,
                            dicom_lookup: dict) -> pd.DataFrame:
    """Remove linhas cujo arquivo de imagem não existe de fato no disco,
    APÓS a tradução via dicom_info.csv. Isso evita que o treino inteiro
    trave por causa de algumas linhas com dados inconsistentes no CSV
    original (comum em datasets grandes reais) — em vez disso, avisa
    quantas/quais foram descartadas e segue com as válidas."""
    resolved_paths = []
    exists_flags = []

    for raw_path in df[image_path_column]:
        resolved = resolve_image_path(dataset_root, raw_path, dicom_lookup)
        resolved_paths.append(resolved)
        exists_flags.append(os.path.exists(resolved))

    df = df.copy()
    df["_resolved_path"] = resolved_paths
    df["_path_exists"] = exists_flags

    n_before = len(df)
    missing_df = df[~df["_path_exists"]]
    df = df[df["_path_exists"]].reset_index(drop=True)
    n_after = len(df)

    if n_after < n_before:
        n_missing = n_before - n_after
        pct = 100 * n_missing / n_before
        print(f"[dataset] AVISO: {n_missing} de {n_before} linhas ({pct:.1f}%) descartadas "
              f"por não terem arquivo de imagem correspondente no disco "
              f"(inconsistência entre o CSV de casos e o dicom_info.csv/pasta jpeg/).")
        if n_missing <= 10:
            print("[dataset] Pacientes afetados:", missing_df["patient_id"].tolist())
        if pct > 5:
            print(f"[dataset] ATENÇÃO: {pct:.1f}% é uma proporção alta — vale investigar "
                  f"antes de confiar nos resultados (pode ser um problema sistemático, "
                  f"não só ruído aleatório).")

    return df.drop(columns=["_resolved_path", "_path_exists"])


def patient_level_split(df: pd.DataFrame, val_fraction: float, seed: int):
    rng = np.random.RandomState(seed)
    patients = df["patient_id"].unique()
    rng.shuffle(patients)

    n_val = max(1, int(len(patients) * val_fraction))
    val_patients = set(patients[:n_val])

    val_df = df[df["patient_id"].isin(val_patients)].reset_index(drop=True)
    train_df = df[~df["patient_id"].isin(val_patients)].reset_index(drop=True)

    return train_df, val_df


### Métricas e reprodutibilidade (seeds, AUROC/AUPRC, bootstrap CI)

In [ ]:
# utils.py
"""Utilidades de reprodutibilidade e métricas (AUROC, AUPRC, sensibilidade/
especificidade em ponto de operação fixo, com intervalos de confiança de 95%
por bootstrap), conforme exigido no protocolo experimental do projeto."""

import random

import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, f1_score,
)


def set_seed(seed: int):
    """Fixa a semente em todas as fontes de aleatoriedade relevantes."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo adicional (pode reduzir performance um pouco, mas é mais
    # importante para reprodutibilidade em um projeto acadêmico):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sensitivity_at_specificity(y_true, y_score, target_specificity=0.90):
    """Retorna (sensibilidade, limiar) no ponto em que a especificidade é
    a mais próxima possível (por cima) do valor alvo (padrão: 90%)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    valid = specificity >= target_specificity
    if not valid.any():
        # nenhum ponto atinge a especificidade alvo -> usa o mais próximo
        idx = np.argmax(specificity)
    else:
        # entre os pontos que atingem a especificidade alvo, pega o de
        # maior sensibilidade (tpr)
        candidate_idx = np.where(valid)[0]
        idx = candidate_idx[np.argmax(tpr[candidate_idx])]
    return float(tpr[idx]), float(thresholds[idx])


def specificity_at_sensitivity(y_true, y_score, target_sensitivity=0.90):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    valid = tpr >= target_sensitivity
    if not valid.any():
        idx = np.argmax(tpr)
    else:
        candidate_idx = np.where(valid)[0]
        idx = candidate_idx[np.argmax(specificity[candidate_idx])]
    return float(specificity[idx]), float(thresholds[idx])


def compute_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    sens90, thr_sens90 = sensitivity_at_specificity(y_true, y_score, 0.90)
    spec90, thr_spec90 = specificity_at_sensitivity(y_true, y_score, 0.90)
    cm = confusion_matrix(y_true, y_pred).tolist()
    f1 = f1_score(y_true, y_pred)

    return {
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity_at_specificity_90": sens90,
        "specificity_at_sensitivity_90": spec90,
        "f1_at_threshold_0.5": f1,
        "confusion_matrix_at_threshold_0.5": cm,
        "n_samples": len(y_true),
        "n_positive": int(y_true.sum()),
    }


def bootstrap_ci(y_true, y_score, metric_fn, n_bootstrap=1000, ci=0.95, seed=42):
    """Intervalo de confiança por bootstrap (reamostragem com reposição),
    conforme exigido no protocolo (mínimo 1000 reamostragens)."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)

    values = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        y_true_bs = y_true[idx]
        y_score_bs = y_score[idx]
        if len(np.unique(y_true_bs)) < 2:
            continue  # bootstrap sem as duas classes -> métrica indefinida, pula
        values.append(metric_fn(y_true_bs, y_score_bs))

    values = np.array(values)
    alpha = 1 - ci
    lower = float(np.percentile(values, 100 * alpha / 2))
    upper = float(np.percentile(values, 100 * (1 - alpha / 2)))
    return lower, upper, float(np.mean(values))


def full_metrics_with_ci(y_true, y_score, threshold=0.5, n_bootstrap=1000, seed=42):
    """Métricas completas + IC 95% por bootstrap para AUROC e AUPRC."""
    base = compute_metrics(y_true, y_score, threshold=threshold)

    auroc_lo, auroc_hi, auroc_mean = bootstrap_ci(
        y_true, y_score, roc_auc_score, n_bootstrap=n_bootstrap, seed=seed
    )
    auprc_lo, auprc_hi, auprc_mean = bootstrap_ci(
        y_true, y_score, average_precision_score, n_bootstrap=n_bootstrap, seed=seed
    )

    base["auroc_ci95"] = [auroc_lo, auroc_hi]
    base["auprc_ci95"] = [auprc_lo, auprc_hi]
    base["bootstrap_n"] = n_bootstrap

    return base


### Conexão ao MLflow

In [ ]:
# ============================================================
# Instalação e conexão ao MLflow.
# A URL/credencial do servidor MLflow NUNCA vai hardcoded aqui — vem de
# Kaggle Secrets (Add-ons > Secrets no menu do notebook), conforme exigido
# pelo critério de aceite (nenhuma credencial publicada no notebook).
# Configuração do servidor em si é escopo da MLOPS-02, não deste notebook.
# ============================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow"], check=True)

import mlflow

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    MLFLOW_TRACKING_URI = secrets.get_secret("MLFLOW_TRACKING_URI")
    # Se o servidor exigir usuário/senha ou token, defina também como secrets
    # e exporte como variável de ambiente aqui (nunca como texto literal):
    if secrets.get_secret("MLFLOW_TRACKING_TOKEN"):
        os.environ["MLFLOW_TRACKING_TOKEN"] = secrets.get_secret("MLFLOW_TRACKING_TOKEN")
except Exception:
    # Fora do Kaggle (ex: rodando local para debug), usa variável de ambiente
    MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")

if not MLFLOW_TRACKING_URI:
    raise RuntimeError(
        "MLFLOW_TRACKING_URI não configurada. Adicione como Kaggle Secret "
        "(Add-ons > Secrets) antes de rodar — não hardcode a URL aqui."
    )

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Nome do experimento segue a convenção <grupo>-<tarefa>-<eixo-de-variacao>
EXPERIMENT_NAME_SMOKE = "g1-classificacao-smoke-test"
EXPERIMENT_NAME_FULL = f"g1-classificacao-{CONFIG['ablation_axis']}"

mlflow.set_experiment(EXPERIMENT_NAME_SMOKE)
print(f"Conectado ao MLflow em: {MLFLOW_TRACKING_URI}")
print(f"Experimento (smoke test): {EXPERIMENT_NAME_SMOKE}")
print(f"Experimento (run completo): {EXPERIMENT_NAME_FULL}")


### Modelo e lógica de execução de um run (compartilhada entre smoke test e run completo)

In [ ]:
import torch
import torch.nn as nn
import subprocess
from torch.utils.data import DataLoader
from torchvision import models

# Caminho do script de avaliação compartilhado do grupo — ajuste se o seu
# dataset Kaggle tiver um nome diferente.
EVALUATE_PATH = "/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado/evaluate_shared/evaluate_shared"


def build_model(architecture, pretrained, num_classes=1):
    if architecture != "resnet50":
        raise NotImplementedError(f"Arquitetura '{architecture}' não implementada.")
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels, _pid, _path in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().flatten().tolist())
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
    metrics = compute_metrics(all_labels, all_scores)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def run_shared_evaluate(predictions_df: "pd.DataFrame", run_id: str) -> dict:
    """Chama o evaluate.py compartilhado do grupo em vez de calcular as
    métricas com lógica própria — garante que a avaliação usa exatamente
    o mesmo protocolo que os outros grupos/sprints, sem escolher limiar
    no teste (isso é responsabilidade do evaluate.py, não daqui)."""
    eval_input_df = predictions_df.rename(
        columns={"patient_id": "case_id", "true_label": "y_true", "predicted_probability": "y_score"}
    )[["case_id", "y_true", "y_score"]]

    preds_path = os.path.join(WORKDIR, f"{run_id}_preds_for_eval.csv")
    metrics_out_path = os.path.join(WORKDIR, f"{run_id}_eval_metrics.json")
    eval_input_df.to_csv(preds_path, index=False)

    cmd = ["python", f"{EVALUATE_PATH}/evaluate.py",
           "--task", "classification", "--input", preds_path, "--output", metrics_out_path]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0 or not os.path.exists(metrics_out_path):
        print("[evaluate.py] FALHOU — stdout/stderr abaixo. Usando métricas internas como fallback.")
        print(result.stdout)
        print(result.stderr)
        os.remove(preds_path)
        return None

    with open(metrics_out_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

    # Loga o CSV de entrada e o metrics.json de saída como artefatos, para
    # rastreabilidade completa de qual avaliação gerou qual número.
    mlflow.log_artifact(preds_path)
    mlflow.log_artifact(metrics_out_path)
    os.remove(preds_path)
    os.remove(metrics_out_path)

    return metrics


def log_metrics_dict_to_mlflow(metrics: dict, prefix: str = ""):
    """Loga qualquer dict de métricas no MLflow, achatando valores aninhados
    (ex: listas de intervalo de confiança viram _low/_high). Genérico de
    propósito, pois não sabemos de antemão todas as chaves exatas que o
    evaluate.py compartilhado retorna."""
    for key, value in metrics.items():
        full_key = f"{prefix}{key}"
        if isinstance(value, bool):
            continue
        elif isinstance(value, (int, float)):
            mlflow.log_metric(full_key, value)
        elif isinstance(value, (list, tuple)) and len(value) == 2 and all(isinstance(v, (int, float)) for v in value):
            mlflow.log_metric(f"{full_key}_low", value[0])
            mlflow.log_metric(f"{full_key}_high", value[1])
        elif isinstance(value, dict):
            log_metrics_dict_to_mlflow(value, prefix=f"{full_key}_")
        # strings, listas de tamanho != 2, matrizes de confusão completas etc.
        # ficam só no metrics.json (artefato), não como mlflow.log_metric.


def execute_run(config: dict, smoke_test: bool, mlflow_experiment: str,
                 max_smoke_samples: int = 16, extra_tags: dict = None):
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"

    mlflow.set_experiment(mlflow_experiment)

    with mlflow.start_run(run_name=f"{config['experiment_id']}{'-smoke' if smoke_test else ''}") as run:
        mlflow.set_tags({
            "owner": config["owner"],
            "role": config["role"],
            "status": "exploratory" if smoke_test else "success",  # sobrescrito no fim/erro
            "ablation_axis": "none" if smoke_test else config["ablation_axis"],
            "environment": ENVIRONMENT,
            **(extra_tags or {}),
        })

        mlflow.log_params({
            "code_version": CODE_VERSION,
            "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"],
            "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"],
            "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"],
            "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"],
            "preprocessing_version": config["preprocessing_version"],
            "environment": ENVIRONMENT,
        })

        try:
            print("Construindo mapa de tradução dicom_info.csv -> jpeg real...")
            dicom_lookup = build_dicom_info_lookup(config["dataset_root"])
            print(f"Mapa construído: {len(dicom_lookup)} imagens mapeadas.")

            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])

            if smoke_test:
                train_df = train_df.head(max_smoke_samples)
                val_df = val_df.head(max(4, max_smoke_samples // 4))

            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=not smoke_test,
                                             dicom_lookup=dicom_lookup)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup)

            # num_workers>0 paraleliza a leitura/decodificação de imagem, que é
            # o gargalo real (não a GPU) — sem isso a GPU fica ociosa esperando
            # dado. pin_memory acelera a transferência CPU->GPU quando há CUDA.
            NUM_WORKERS = 2
            train_loader = DataLoader(train_ds, batch_size=min(config["batch_size"], len(train_ds)),
                                       shuffle=True, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                       persistent_workers=NUM_WORKERS > 0)
            val_loader = DataLoader(val_ds, batch_size=min(config["batch_size"], len(val_ds)),
                                     shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                     persistent_workers=NUM_WORKERS > 0)

            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            n_epochs = 1 if smoke_test else config["epochs"]
            history = []
            t_start = time.time()
            best_val_auroc, checkpoint_path = 0.0, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")

            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:
                    torch.cuda.reset_peak_memory_stats(device)

                train_m = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                val_m = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0

                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None

                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("train_auroc", train_m["auroc"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                if peak_vram_mb is not None:
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)

                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "train_auroc": train_m["auroc"], "val_auroc": val_m["auroc"],
                                 "seconds": dt, "peak_vram_mb": peak_vram_mb})

                vram_str = f"{peak_vram_mb:.0f}MB" if peak_vram_mb is not None else "N/A (CPU)"
                print(f"Época {epoch}/{n_epochs} | val_auroc={val_m['auroc']:.4f} | {dt:.1f}s | VRAM pico: {vram_str}")

                if val_m["auroc"] > best_val_auroc:
                    best_val_auroc = val_m["auroc"]
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "image_size": config["image_size"], "seed": config["seed"],
                                "experiment_id": config["experiment_id"]}, checkpoint_path)

            duration_seconds = sum(h["seconds"] for h in history)
            avg_epoch_seconds = duration_seconds / len(history) if history else None
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]

            mlflow.log_metric("duration_seconds", duration_seconds)
            if avg_epoch_seconds is not None:
                mlflow.log_metric("avg_epoch_time_seconds", avg_epoch_seconds)
            if vram_values:
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))

            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)

            config_path = os.path.join(WORKDIR, f"{run.info.run_id}_config.json")
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(config, f, indent=2, ensure_ascii=False)
            mlflow.log_artifact(config_path)
            os.remove(config_path)

            if not smoke_test:
                test_df = build_labeled_dataframe(
                    config["dataset_root"], config["test_csvs"], config["malignant_labels"],
                    config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
                )
                test_df = filter_resolvable_rows(test_df, config["dataset_root"],
                                                  config["image_path_column"], dicom_lookup)
                test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                                config["image_size"], mean, std, augment=False,
                                                dicom_lookup=dicom_lookup)
                test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                                          num_workers=NUM_WORKERS, pin_memory=use_cuda)

                model.eval()
                records = []
                with torch.no_grad():
                    for images, labels, patient_ids, _paths in test_loader:
                        probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
                        for pid, lab, prob in zip(patient_ids, labels.numpy().flatten(), probs):
                            records.append({"patient_id": pid, "true_label": int(lab),
                                             "predicted_probability": float(prob)})

                predictions_df = pd.DataFrame(records)

                # ---- Avaliação via evaluate.py compartilhado (protocolo único do grupo) ----
                print("Rodando evaluate.py compartilhado...")
                shared_metrics = run_shared_evaluate(predictions_df, run.info.run_id)

                if shared_metrics is not None:
                    log_metrics_dict_to_mlflow(shared_metrics)
                    mlflow.set_tag("evaluation_source", "evaluate_shared")
                    print("Métricas do evaluate.py compartilhado registradas no MLflow.")
                else:
                    # Fallback: métricas internas, só se o script compartilhado falhar
                    metrics_test = full_metrics_with_ci(
                        predictions_df["true_label"].values, predictions_df["predicted_probability"].values,
                        n_bootstrap=1000, seed=config["seed"],
                    )
                    log_metrics_dict_to_mlflow(metrics_test)
                    mlflow.set_tag("evaluation_source", "internal_fallback")
                    print("[AVISO] evaluate.py compartilhado falhou — usadas métricas internas como fallback.")

                if os.path.exists(checkpoint_path):
                    mlflow.log_artifact(checkpoint_path)
                    os.remove(checkpoint_path)

            else:
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)

            mlflow.set_tag("status", "success")
            return run.info.run_id

        except Exception as e:
            # Runs falhos também ficam registrados (não são descartados nem
            # escondidos), conforme critério de aceite da ablação.
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            print(f"[ERRO] Run falhou, mas foi registrado no MLflow com status=failed: {e}")
            raise


## Smoke test (obrigatório antes do run completo)

Roda uma versão reduzida (poucas imagens, 1 época) só para validar que o
pipeline inteiro funciona neste ambiente — dataset carrega, modelo treina,
MLflow registra. Tag `status=exploratory`, fica isolado no experimento
`g1-classificacao-smoke-test`, nunca misturado com resultados de comparação
real.

In [ ]:
print("=== SMOKE TEST — validando o pipeline antes do run completo ===")
smoke_run_id = execute_run(
    CONFIG, smoke_test=True, mlflow_experiment=EXPERIMENT_NAME_SMOKE,
    max_smoke_samples=16,
)
print(f"\nSmoke test concluído com sucesso. Run ID: {smoke_run_id}")
print("Pipeline validado — pode prosseguir para o run completo na próxima célula.")


## Run completo

**Só rode esta célula depois que o smoke test acima terminou sem erro.**
Treina com a configuração completa de `CONFIG`, avalia no teste oficial
congelado, e registra tudo no MLflow (experimento
`g1-classificacao-<eixo>`).

In [ ]:
# Só chegue até aqui depois que a célula do smoke test rodou sem erro.
print("=== RUN COMPLETO ===")
full_run_id = execute_run(
    CONFIG, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_FULL,
)
print(f"\nRun completo concluído. Run ID: {full_run_id}")
print(f"Confira os resultados no MLflow, experimento '{EXPERIMENT_NAME_FULL}'.")


**Batch Size 4 configs**

**Treino com os batch de 6, 8, 16 etc**

In [ ]:
# ============================================================
# Configurações versionadas da ablação de batch size.
# Cada uma é uma cópia EXATA do CONFIG do EXP-01 (baseline), mudando
# APENAS batch_size e experiment_id — nenhuma outra variável muda,
# conforme o critério de aceite desta sprint.
# ============================================================

CONFIG_BATCH_08 = {**CONFIG, "experiment_id": "EXP-13", "batch_size": 8, "ablation_axis": "batch_size"}
CONFIG_BATCH_16 = {**CONFIG, "experiment_id": "EXP-14", "batch_size": 16, "ablation_axis": "batch_size"}
CONFIG_BATCH_32 = {**CONFIG, "experiment_id": "EXP-15", "batch_size": 32, "ablation_axis": "batch_size"}
CONFIG_BATCH_64 = {**CONFIG, "experiment_id": "EXP-16", "batch_size": 64, "ablation_axis": "batch_size"}

BATCH_CONFIGS = [CONFIG_BATCH_08, CONFIG_BATCH_16, CONFIG_BATCH_32, CONFIG_BATCH_64]

# Conferência rápida: só batch_size deve diferir entre eles
campos_para_conferir = ["seed", "architecture", "pretrained_weights", "image_size",
                         "learning_rate", "weight_decay", "epochs", "dataset_root"]
for campo in campos_para_conferir:
    valores = {c["experiment_id"]: c[campo] for c in BATCH_CONFIGS}
    if len(set(valores.values())) > 1:
        raise RuntimeError(f"Campo '{campo}' difere entre os configs: {valores} "
                            f"— isso quebra o critério de 'só batch size muda'.")
print("Conferência OK: todos os configs são idênticos exceto batch_size.")
for c in BATCH_CONFIGS:
    print(f"  {c['experiment_id']}: batch_size={c['batch_size']}")


In [ ]:
# ============================================================
# Roda os 4 experimentos da ablação. Cada um em seu próprio try/except
# — se um falhar, os outros continuam rodando (runs falhos ficam
# registrados no MLflow com status=failed, não travam a sprint inteira).
# ============================================================

EXPERIMENT_NAME_BATCH = "g1-classificacao-batch_size"
batch_run_ids = {}

for cfg in BATCH_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Rodando {cfg['experiment_id']} (batch_size={cfg['batch_size']})")
    print(f"{'='*60}")
    try:
        run_id = execute_run(cfg, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_BATCH)
        batch_run_ids[cfg["experiment_id"]] = {"run_id": run_id, "batch_size": cfg["batch_size"], "status": "success"}
        print(f"OK — {cfg['experiment_id']} concluído. run_id: {run_id}")
    except Exception as e:
        batch_run_ids[cfg["experiment_id"]] = {"run_id": None, "batch_size": cfg["batch_size"],
                                                 "status": "failed", "error": str(e)[:200]}
        print(f"FALHOU — {cfg['experiment_id']} (batch_size={cfg['batch_size']}): {e}")
        print("Registrado no MLflow com status=failed. Seguindo para o próximo batch size.")

print("\n\n=== RESUMO DA ABLAÇÃO ===")
for exp_id, info in batch_run_ids.items():
    print(f"  {exp_id} (batch={info['batch_size']}): {info['status']} | run_id: {info['run_id']}")


In [ ]:
# ============================================================
# Manifesto consolidado da ablação — reúne tudo que a issue de
# "coleta de resultados" pede num único arquivo exportável, sem
# precisar que quem avalia entre no MLflow run por run.
# ============================================================

client = mlflow.tracking.MlflowClient()

manifest_rows = []
for exp_id, info in batch_run_ids.items():
    row = {
        "experiment_id": exp_id,
        "run_id": info["run_id"] or "N/A",
        "status": info["status"],
        "batch_size": info["batch_size"],
    }

    if info["run_id"]:
        run = client.get_run(info["run_id"])
        p = run.data.params
        m = run.data.metrics
        row.update({
            # commit/configuração
            "code_version": p.get("code_version", "NAO DEFINIDO"),
            "seed": p.get("seed"),
            "architecture": p.get("architecture"),
            "pretrained_weights": p.get("pretrained_weights"),
            "image_size": p.get("image_size"),
            "learning_rate": p.get("learning_rate"),
            "weight_decay": p.get("weight_decay"),
            "epochs": p.get("epochs"),
            # dataset/manifest e split
            "dataset_manifest": p.get("dataset_manifest"),
            "split_strategy": p.get("split_strategy"),
            # ambiente
            "environment": p.get("environment"),
            "gpu_name": p.get("gpu_name", "N/A"),
            # custo
            "avg_epoch_time_seconds": m.get("avg_epoch_time_seconds"),
            "peak_vram_mb_max": m.get("peak_vram_mb_max"),
            "duration_seconds": m.get("duration_seconds"),
            # avaliação
            "evaluation_source": run.data.tags.get("evaluation_source", "N/A"),
            "auroc": m.get("auroc"),
            # artefatos externos referenciados (não embutidos, só a referência)
            "dataset_source_kaggle": CONFIG["dataset_root"],
            "evaluate_script_path": EVALUATE_PATH,
        })
    else:
        row["error_message"] = info.get("error", "")

    manifest_rows.append(row)

manifest_df = pd.DataFrame(manifest_rows).sort_values("batch_size")

# Aviso explícito se algum code_version não estiver definido — é critério
# de aceite desta sprint, não pode passar batido.
sem_code_version = manifest_df[manifest_df.get("code_version", "") == "NAO DEFINIDO"]
if len(sem_code_version) > 0:
    print("[ATENÇÃO] Os seguintes runs NÃO têm code_version definido — "
          "corrija antes de considerar a sprint concluída:")
    print(sem_code_version["experiment_id"].tolist())

manifest_path = os.path.join(WORKDIR, "manifesto_resultados_batch_ablation.csv")
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifesto salvo em: {manifest_path}")
print(manifest_df.to_string(index=False))

# Registra o manifesto como artefato também, num run "documentação" próprio
mlflow.set_experiment(EXPERIMENT_NAME_BATCH)
with mlflow.start_run(run_name="EXP-batch-ablation-manifesto"):
    mlflow.set_tags({"owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
                      "run_type": "results_manifest", "environment": ENVIRONMENT})
    mlflow.log_artifact(manifest_path)

print("\nManifesto também registrado no MLflow (run 'EXP-batch-ablation-manifesto').")
